In [1]:
import xarray as xr, netCDF4 as nc, numpy as np, pandas as pd, os
from pathlib import Path
from scipy.spatial import cKDTree

In [2]:
os.chdir('/g/data/ng72/ms5578/ID_HW_BARRA')
working_dir = Path().absolute()

ehf_fpath = '/scratch/ng72/ms5578'
nmap_path = '/g/data/ng72/ms5578/ID_HW_BARRA/data/raw'
write_path = '/scratch/ng72/ms5578/time_series'

In [3]:
sdate, edate = "2018-01-08 00:00:00", "2019-01-28 23:59:59"

In [4]:
yr_file = f"{ehf_fpath}/hw_files/HW_EHF_2018_2019.nc"
ds = xr.open_dataset(yr_file,
                         engine='netcdf4',
                         chunks="auto")

EHF_ds = ds.sel(time=slice(sdate, edate))

In [5]:
gen_df = pd.read_csv(f"{nmap_path}/nmap.csv")
gen_df.drop(gen_df.columns[[3, 1, 4, 5]], axis=1, inplace=True)

gen_df.columns = (
    gen_df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace(r'[^\w_]', '', regex=True)
    .str.replace('__', '_')
)

First we join the generator information with the EHF info, sticking to daily intervals

In [6]:
def merge_nearest(ds,df):
    # Extract latitude and longitude grids
    lat_grid = ds['lat'].values
    lon_grid = ds['lon'].values
    
    # Convert gen_df lat/lon to numpy arrays for vectorized operations
    gen_lats = df['lat'].values
    gen_lons = df['lon'].values
    
    # Find the nearest indices for all latitudes and longitudes in gen_df
    lat_indices = np.abs(lat_grid[:, None] - gen_lats).argmin(axis=0)
    lon_indices = np.abs(lon_grid[:, None] - gen_lons).argmin(axis=0)
    
    # Get the nearest latitudes and longitudes
    nearest_lats = lat_grid[lat_indices]
    nearest_lons = lon_grid[lon_indices]
    
    # Create DataArrays for vectorized selection
    nearest_lats_da = xr.DataArray(nearest_lats, dims="DUID")
    nearest_lons_da = xr.DataArray(nearest_lons, dims="DUID")
    
    new_ds = ds.sel(lat=nearest_lats_da, lon=nearest_lons_da)
    new_ds = new_ds.assign_coords(DUID=df['duid'])

    return new_ds

hw_info = merge_nearest(EHF_ds, gen_df).to_dataframe().reset_index()
hw_info = hw_info.replace(1.000000e+20, np.nan)
hw_info.to_csv(f"{write_path}/gen_hw_status.csv")
ds.close()